In [19]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
chunks = [
  "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",
  "Tesla Cybertruck production ramp begins in 2024.",
  "Google is a large technology company with global operations.",
  "Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.",
  "SpaceX develops Starship rockets for Mars missions.",
  "The tech giant acquired the code repository platform for software development.",
  "NVIDIA designs Starship architecture for their new GPUs.",
  "Tesla Tesla Tesla financial quarterly results improved significantly.",
  "Cybertruck reservations exceeded company expectations.",
  "Microsoft is a large technology company with global operations.", 
  "Apple announced new iPhone features for developers.",
  "The apple orchard harvest was excellent this year.",
  "Python programming language is widely used in AI.",
  "The python snake can grow up to 20 feet long.",
  "Java coffee beans are imported from Indonesia.", 
  "Java programming requires understanding of object-oriented concepts.",
  "Orange juice sales increased during winter months.",
  "Orange County reported new housing developments."
]

In [5]:
documents = [Document(page_content=chunk, metadata={"source": f"Chunk_{i}"}) for i, chunk in enumerate(chunks)]

print("Sampel data:")
for i, chunk in enumerate(documents, 1):
  print(f"{i}. {chunk.page_content}")

Sampel data:
1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Tesla Cybertruck production ramp begins in 2024.
3. Google is a large technology company with global operations.
4. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
5. SpaceX develops Starship rockets for Mars missions.
6. The tech giant acquired the code repository platform for software development.
7. NVIDIA designs Starship architecture for their new GPUs.
8. Tesla Tesla Tesla financial quarterly results improved significantly.
9. Cybertruck reservations exceeded company expectations.
10. Microsoft is a large technology company with global operations.
11. Apple announced new iPhone features for developers.
12. The apple orchard harvest was excellent this year.
13. Python programming language is widely used in AI.
14. The python snake can grow up to 20 feet long.
15. Java coffee beans are imported from Indonesia.
16. Java pr

# 1. Vector Retriever (Semantics Search/Dense Retrieval)

In [7]:
print("Setting up the vector retriever....")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(
  documents=documents,
  embedding=embedding_model,
  collection_metadata={"hnsw:space": "cosine"}
)

Setting up the vector retriever....


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2191.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "space exploration company"

print(f"Test query: {test_query}")
resultant_docs = vector_retriever.invoke(test_query)

print("Resultant docs:")
for i, doc in enumerate(resultant_docs, 1):
  print(f"{i}. {doc.page_content}")

Test query: space exploration company
Resultant docs:
1. SpaceX develops Starship rockets for Mars missions.
2. Google is a large technology company with global operations.
3. Microsoft is a large technology company with global operations.


# 2. BM25 Retriever (Keyword Search/Sparse Retrieval)

In [9]:
print("Setting up the BM25 Retriever....")
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 3

Setting up the BM25 Retriever....


In [10]:
# Test exact keyword matching
# test_query = "space exploration company"
test_query = "Cybertruck"
# test_query = "Tesla"

print(f"Test query: {test_query}")
bm25_search_docs = bm25_retriever.invoke(test_query)
for i, doc in enumerate(bm25_search_docs, 1):
  print(f"{i}. {doc.page_content}")

Test query: Cybertruck
1. Cybertruck reservations exceeded company expectations.
2. Tesla Cybertruck production ramp begins in 2024.
3. Orange juice sales increased during winter months.


# 3. Hybrid Retriever (Combination)

In [15]:
print("Setting up Hybrid Retriever...")
hybrid_retriever = EnsembleRetriever(
  retrievers=[vector_retriever, bm25_retriever],
  weights=[0.5, 0.5]
)
print("Setup complete")

Setting up Hybrid Retriever...
Setup complete


In [16]:
test_query = "purchase cost 7.5 billion"

retrieved_chunks = hybrid_retriever.invoke(test_query)
print(f"Test query: {test_query}")
for i, doc in enumerate(retrieved_chunks, 1):
  print(f"{i}. {doc.page_content}")

Test query: purchase cost 7.5 billion
1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Microsoft is a large technology company with global operations.
3. Orange County reported new housing developments.
4. Cybertruck reservations exceeded company expectations.
5. Orange juice sales increased during winter months.


In [17]:
test_query = "electric vehicle manufacturing Cybertruck"

retrieved_chunks = hybrid_retriever.invoke(test_query)
print(f"Test query: {test_query}")
for i, doc in enumerate(retrieved_chunks, 1):
  print(f"{i}. {doc.page_content}")

Test query: electric vehicle manufacturing Cybertruck
1. Tesla Cybertruck production ramp begins in 2024.
2. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
3. Cybertruck reservations exceeded company expectations.


In [18]:
test_query = "company performance Tesla"

retrieved_chunks = hybrid_retriever.invoke(test_query)
print(f"Test query: {test_query}")
for i, doc in enumerate(retrieved_chunks, 1):
  print(f"{i}. {doc.page_content}")

Test query: company performance Tesla
1. Tesla Tesla Tesla financial quarterly results improved significantly.
2. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
3. Tesla Cybertruck production ramp begins in 2024.
4. Cybertruck reservations exceeded company expectations.


In [20]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI

combined_input = f"""Based on the following documents, please answer this question: {test_query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in retrieved_chunks])}

Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say "I don't have enough information to answer that question based on the provided documents."
"""

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

messages = [
  SystemMessage(content="You are a helpful assistant."),
  HumanMessage(content=combined_input)
]

result = llm.invoke(messages)

print("---Generated Response---")
print(result.content)

---Generated Response---
Based on the provided documents, Tesla's company performance can be described as:

*   Financial quarterly results improved significantly and were strong.
*   Tesla continues to lead in electric vehicles.
*   Cybertruck reservations exceeded company expectations.
